In [ ]:
import pandas as pd
import numpy as np  
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import os
import joblib
from pathlib import Path

In [ ]:
load_dotenv()

In [ ]:
COUNTRY_CODE = os.getenv("COUNTRY_CODE")
RESULTS_SAVE_FOLDER = os.getenv("RESULTS_SAVE_FOLDER")
BASE_DATA_PATH = os.getenv("BASE_DATA_PATH")

In [ ]:
def calculate_features(df):
   
    df_engineered = df.copy()
    df_engineered['Samplingpoint'] = df_engineered['Samplingpoint'].astype('category')
    df_engineered['DayOfWeek'] = df_engineered['Start'].dt.dayofweek
    df_engineered['Month'] = df_engineered['Start'].dt.month
    df_engineered['DayOfYear'] = df_engineered['Start'].dt.dayofyear
    df_engineered['IsWeekend'] = df_engineered['DayOfWeek'].isin([5, 6]).astype(int)
    df_engineered['Lag_1'] = df_engineered.groupby('Samplingpoint')['Value'].shift(1) 
    df_engineered['Lag_7'] = df_engineered.groupby('Samplingpoint')['Value'].shift(7) 

    df_engineered = df_engineered.set_index('Start') # Need index to be the 'Start'(date) column for rolling with time-based windows to work properly

    df_engineered['Rolling_Mean_7D'] = df_engineered.groupby('Samplingpoint')['Value'].transform(
    lambda x: x.rolling('7D', min_periods=4).mean() # 7d creates rolling window of 7 calendar days instead of last 7 rows
    )

    df_engineered = df_engineered.reset_index()

    df_engineered['Target'] = df_engineered.groupby('Samplingpoint')['Value'].shift(-1)

    df_engineered = df_engineered.dropna(subset=['Target']).reset_index(drop=True)



    return df_engineered


In [ ]:
def load_and_predict(pollutant):
    
    file_name = f"{COUNTRY_CODE}_{pollutant}_Daily_Cleaned.parquet"
    file_path = os.path.join(BASE_DATA_PATH, file_name)
    
    RUN_ID = f"{COUNTRY_CODE}_{pollutant}"
    output_dir = Path(RESULTS_SAVE_FOLDER) / f"Results_{RUN_ID}"
    
    model_filename = output_dir / f"xgboost_model_{RUN_ID}.pkl"
    features_filename = output_dir / f"xgboost_features_{RUN_ID}.pkl"
    
    model = joblib.load(model_filename)
    features = joblib.load(features_filename)

    df = pd.read_parquet(file_path, engine='pyarrow')

    df = calculate_features(df)
    

    test_data = df[(df['Start'] >= '2023-01-01') & (df['Start'] <= '2024-12-31')].copy()
    test_data = test_data.sort_values(by='Start').reset_index(drop=True)
    
    X_test = test_data[features]
    test_preds = model.predict(X_test)
    
    test_data['Predictions'] = test_preds
    
    print(f"Successfully generated {len(test_preds)} predictions for {pollutant}.\n")
    
    return {
        "model": model,
        "features": features,
        "results_df": test_data[['Start', 'Target', 'Predictions']] 
    }

In [ ]:
def plot_results(pollutant, results_df):
    
    # Setup the save directory specifically for this pollutant
    RUN_ID = f"{COUNTRY_CODE}_{pollutant}"
    output_dir = Path(RESULTS_SAVE_FOLDER) / f"Results_{RUN_ID}"
    pred_plot_dir = output_dir / f"{COUNTRY_CODE}_{pollutant}_Global_Predict_vs_Actual_Plots"
    pred_plot_dir.mkdir(parents=True, exist_ok=True)

  
    df_plot = results_df.copy()
    df_plot = df_plot.rename(columns={'Target': 'Actual', 'Predictions': 'Predicted'})
    df_plot['Start'] = pd.to_datetime(df_plot['Start'])

    # Calc national daily average
    national_df = df_plot.groupby('Start')[['Actual', 'Predicted']].mean().reset_index().sort_values('Start')
    
    max_date = national_df['Start'].max()
    last_1y = national_df[national_df['Start'] >= max_date - pd.DateOffset(years=1)]
    last_2y = national_df[national_df['Start'] >= max_date - pd.DateOffset(years=2)]

    def plot_national(df, title_suffix, save_name):
        if df.empty:
            print(f"Skipping {title_suffix} - Not enough data.")
            return
        
        rmse = np.sqrt(mean_squared_error(df['Actual'], df['Predicted']))
        mae = mean_absolute_error(df['Actual'], df['Predicted'])
        r2 = r2_score(df['Actual'], df['Predicted'])

        plt.figure(figsize=(16, 5))

        plt.plot(df['Start'], df['Actual'], label=f'Actual {pollutant} (Daily Avg)', color='blue', marker='o', markersize=3, linewidth=1, alpha=0.9)
        plt.plot(df['Start'], df['Predicted'], label=f'Predicted {pollutant} (Daily Avg)', color='red', marker='o', markersize=3, linewidth=1, alpha=0.9)

        plt.title(
            f'National {pollutant} ({title_suffix}) Predictions vs Recorded \n'
            f'RMSE: {rmse:.2f} | MAE: {mae:.2f} | R²: {r2:.2f}'
        )

        plt.xlabel('Date')
        plt.ylabel(f'{pollutant} ug.m-3')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()

        plt.savefig(pred_plot_dir / save_name, dpi=150) 
        plt.show() 
        plt.close()

    plot_national(last_1y, "Last 1 Year", f"{COUNTRY_CODE}_{pollutant}_national_1y.png")
    plot_national(last_2y, "Last 2 Years", f"{COUNTRY_CODE}_{pollutant}_national_2y.png")


In [ ]:
pollutants_list = ["PM10", "PM2.5", "O3", "SO2", "NO2", "NOXasNO2"]

In [ ]:
for pol in pollutants_list:
    result_dict = load_and_predict(pol)
    results_df = result_dict["results_df"]
    plot_results(pol, results_df)